# 4. Parallel specialist reviewers

Multi-agent is useful when work decomposes into independent perspectives. Correctness, security, and maintainability reviewers receive the same immutable artifact and return the same structured contract. Fan-out is capped at three; no reviewer can delegate.

## Before you begin

**Choose one:** use the structured mock reviewer for deterministic classroom work, or OpenRouter when the issued key is available. Never send private code.

### Learning outcomes

Scope three reviewer roles, compare sequential clarity with parallel fan-out, and preserve structured handoffs.

Architecture reference: [Day 4 diagrams D13](../../diagrams/source/day_04.md).

### Expected observation

Three role traces appear and the supervisor receives only Finding objects.

## Concept briefing

## Specialist decomposition

A specialist role should narrow the task, not merely rename the same prompt. Correctness,
security and maintainability reviewers receive the same immutable artifact but different
evaluation criteria. They return the same `Finding` contract: category, location,
evidence, severity and recommended correction.

Structured handoffs prevent unconstrained agent conversations. The supervisor does not
need every reviewer's full chat history. It needs validated findings and enough provenance
to resolve duplicates and conflicts.

## Sequential before parallel

Run specialists sequentially first because the execution order and failures are easy to
inspect. If the branches are independent, they can then fan out in parallel and fan in at
the supervisor. Parallelism may reduce wall-clock time but does not reduce total model
calls or tokens. It may also trigger provider rate limits.

The fan-in step must be bounded. It validates fields, deduplicates, ranks, caps output and
terminates. A supervisor that can indefinitely request revisions has created another
autonomous loop rather than a controlled aggregation step.


In [ ]:
from pathlib import Path
import sys, json
DAY=Path.cwd()
if (DAY/"day_04_multi_agent_systems").exists(): DAY=DAY/"day_04_multi_agent_systems"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"review_team").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
SOURCE=(DAY/"data"/"seeded_artifact"/"order_service.py").read_text(encoding="utf-8")
GOLDEN=DAY/"data"/"golden_defects.json"
print("Artifact lines:",len(SOURCE.splitlines()))

In [ ]:
from review_team import specialist_review
categories=["correctness","security","maintainability"]
groups={category:specialist_review(SOURCE,category) for category in categories}
for category,findings in groups.items():
    print("\n",category)
    for finding in findings: print(finding.as_dict())

## Why parallel?

These branches do not depend on one another, so they may run concurrently and later fan in. Parallelism can reduce wall time with hosted APIs, but raises calls, tokens, rate-limit pressure, and debugging complexity. Local code may be too fast for timing differences to matter.

In [ ]:
import os
from review_team import MockStructuredReviewer,OpenRouterReviewer,run_model_multi
provider=OpenRouterReviewer() if os.getenv("OPENROUTER_API_KEY") else MockStructuredReviewer()
multi=run_model_multi(SOURCE,provider)
print("Calls:",multi.model_calls,"tokens (input estimate):",multi.estimated_tokens)
print("Trace:",multi.trace)

## Optional direct LangGraph

Represent state as artifact plus lists of structured findings. Add three reviewer nodes from `START`, connect all to one supervisor, and compile. Use a reducer for concurrently returned lists. LangGraph coordinates state; it does not make reviewer judgment correct.

## Your turn

Run the same roles sequentially first; then compare calls, results, and wall time with fan-out.

## Recap

Multi-agent means bounded decomposition, not unrestricted agent conversation.